# DLM fine-tuning on Colab (controllable story generation)

Thin runner for `fine-tuning/run_ft.py`; all logic is in `fine-tuning/` and nothing here touches the pre-training code. Run the cells top to bottom with **Runtime -> Change runtime type -> GPU (L4 or A100: bf16)**.

**Needs:** the pre-trained model (Hugging Face repo or a folder on Drive), the committed tokenizer (from the repo), and optionally the pre-training token cache `MyDrive/diffusion_lm_outputs/data/tokens/` (only for the forgetting guard and the gate evaluation).

**The repo on GitHub must contain `fine-tuning/`.** If you have not pushed it yet, upload the folder to `MyDrive/diffusion_lm_outputs/fine-tuning/` (cell 3 falls back to that copy).

Never put `AWS_credential.md` or any token in this notebook; use Colab secrets (`HF_TOKEN`) for private Hub repos.

In [ ]:
# 1. Settings + Drive
INIT_FROM = "bitlabsdb/dlm-tinystories-36m"   # a Hub repo id, or a folder on Drive with config.json + model.pt/safetensors (e.g. .../checkpoints/pretrain_cosine/final)
RUN_NAME = "sft_lr3e-5"       # run folder under checkpoints/. Change it whenever a recipe value below changes: --resume refuses a changed recipe
STEPS = 3000
LR = 3e-5                     # sweep 2e-5 and 5e-5 for ~1500 steps each (different RUN_NAME) before the real run
LORA_R = 0                    # 0 = full fine-tune (recommended); 8 or 32 = the LoRA ablation
KEEP_PROB = 0.2               # share of TinyStoriesInstruct train used (0.2 = about 420k examples)
N_EVAL_PROMPTS = 30           # prompts per condition in the control evaluation
QUIT_WHEN_DONE = False        # True = release the GPU session after the last cell

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/diffusion_lm_outputs"
RUN = f"{OUT}/checkpoints/{RUN_NAME}"
FT_CACHE = f"{OUT}/ft_data/cache"
PRE_CACHE = f"{OUT}/data/tokens"

In [ ]:
# 2. Code from GitHub (public repo; tokenizer/ is committed there)
REPO = "https://github.com/bitlabsdevteam/Diffusion-Language-Model.git"
import os
if os.path.isdir("/content/dlm_repo"):
    !git -C /content/dlm_repo pull --ff-only
else:
    !git clone $REPO /content/dlm_repo
%cd /content/dlm_repo

In [ ]:
# 3. Make sure fine-tuning/ is there (else copy it from Drive)
import os, shutil
if not os.path.isfile("fine-tuning/run_ft.py"):
    src = f"{OUT}/fine-tuning"
    assert os.path.isdir(src), "fine-tuning/ is not on GitHub yet and not on Drive: push it or upload it to " + src
    shutil.copytree(src, "fine-tuning", dirs_exist_ok=True)
!ls fine-tuning

In [ ]:
# 4. Dependencies: install ONLY what is missing (never numpy/torch/transformers: reinstalling them can break Colab's live kernel)
import importlib.util, subprocess, sys
missing = [p for p in ("huggingface_hub", "tokenizers", "safetensors") if importlib.util.find_spec(p) is None]
print("missing:", missing or "none")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

In [ ]:
# 5. Offline tests of the fine-tuning code (about a minute, CPU)
!python fine-tuning/run_ft.py test

In [ ]:
# 6. The pre-trained model: download from the Hub (private repos need the Colab secret HF_TOKEN) or use a Drive folder
import os
if os.path.isdir(INIT_FROM):
    BASE = INIT_FROM
else:
    sys.path.insert(0, "/content/dlm_repo")
    from dlm.hub import download_snapshot
    BASE = download_snapshot(INIT_FROM)
print("base model folder:", BASE)
!ls $BASE

In [ ]:
# 7. STEP 0 GATE: the base model's validation bound must be WELL BELOW the 5.92 nats/token unigram baseline.
# If it is not, stop here: SFT on a model that sits on the token-frequency plateau is wasted GPU time.
# (Cell 10 also enforces this: `train` aborts at step 0 when the plain-prompt bound is above 5.5; override with --gate-uncond-max 0.)
# (needs the pre-training token cache on Drive; skipped with a message otherwise)
import os
if os.path.isfile(f"{PRE_CACHE}/tokens_manifest.json"):
    !python run.py eval --ckpt $BASE --cache $PRE_CACHE --n-stories 20 --n-prompts 5 --no-gpt2 --max-val-blocks 512 --out /content/gate_eval
else:
    print("no pre-training token cache at", PRE_CACHE, "-> gate and forgetting guard are skipped; check the validation bound some other way")

In [ ]:
# 8. Build the SFT cache (TinyStoriesInstruct: parse, strip trailing fields, tokenize). Persisted on Drive, idempotent.
!python fine-tuning/run_ft.py prep --out $FT_CACHE --keep-prob $KEEP_PROB

In [ ]:
# 9. GPU check (an L4/A100 prints "precision bf16"; a T4 falls back to fp32 and is slow)
!nvidia-smi -L

In [ ]:
# 10. Fine-tune (resumable: re-run this cell after a disconnect). Prints a step-0 evaluation of the untouched base model, then EVAL lines
#     with the SFT validation bound and the forgetting guard (plain-prompt bound, drift vs base).
PRE_ARG = f"--pretrain-cache {PRE_CACHE}" if os.path.isfile(f"{PRE_CACHE}/tokens_manifest.json") else ""
!python fine-tuning/run_ft.py train --init-from $BASE --cache $FT_CACHE $PRE_ARG --out $RUN --steps $STEPS --lr $LR --lora-r $LORA_R --resume

In [ ]:
# 11. Curves (SFT validation / train-eval, forgetting guard, learning rate)
from IPython.display import Image, display
for f in (f"{RUN}/final/training_curves.png", f"{RUN}/training_curves.png"):
    if os.path.isfile(f):
        display(Image(f)); break

In [ ]:
# 12. Try it: edit the instruction and run
!python fine-tuning/run_ft.py generate --ckpt $RUN/final --words cat,ball,happy --features Dialogue --n 3
!python fine-tuning/run_ft.py generate --ckpt $RUN/final --sentence "The dog was very brave." --n 2

In [ ]:
# 13. Control evaluation: base vs SFT on the same held-out prompts (report on Drive next to the run)
!python fine-tuning/run_ft.py eval --base $BASE --sft $RUN/final --cache $FT_CACHE --pretrain-cache $PRE_CACHE -n $N_EVAL_PROMPTS --out $RUN/control_eval
print(open(f"{RUN}/control_eval/control_report.md").read())

In [ ]:
# 14. End the session (only when QUIT_WHEN_DONE = True and the final model + report are on Drive)
import os
ok = os.path.isfile(f"{RUN}/final/model.pt") and os.path.isfile(f"{RUN}/control_eval/control_report.json")
if QUIT_WHEN_DONE and ok:
    from google.colab import runtime
    runtime.unassign()
else:
    print("keeping the session (QUIT_WHEN_DONE is False or something is not saved yet)")